In [4]:
import pandas as pd 
import numpy as np
archivo = '../data/raw/exoplanets.csv'
comentarios = []
df = pd.read_csv(archivo, comment='#')


In [5]:
#Selecting only the columns we care about for our analysis
columns_to_keep = [
    'pl_name', 'hostname', 'pl_orbper', 'pl_orbsmax', 
    'pl_rade', 'pl_bmasse', 'pl_insol', 'pl_eqt',
    'st_teff', 'st_rad', 'st_mass', 'discoverymethod',
    'disc_year', 'pl_controv_flag', 'sy_dist'
]

df_clean = df[columns_to_keep].copy()

print(f"New shape: {df_clean.shape}")
print(f"\nMissing per column (remaining):")
missing_clean = df_clean.isnull().sum()
print(missing_clean[missing_clean > 0].sort_values(ascending=False))

# percentage of missing values per row
rows_with_no_missing = (df_clean.isnull().sum(axis=1) == 0).sum()
print(f"\nRows with ZERO missing values: {rows_with_no_missing} ({100*rows_with_no_missing/len(df_clean):.1f}%)")

# Rows if we allow up to 2 missing values
rows_with_max_2_missing = (df_clean.isnull().sum(axis=1) <= 2).sum()
print(f"Rows with ≤2 missing values: {rows_with_max_2_missing} ({100*rows_with_max_2_missing/len(df_clean):.1f}%)")

New shape: (6278, 15)

Missing per column (remaining):
pl_insol      1878
pl_eqt        1601
pl_orbsmax     428
pl_orbper      340
st_rad         318
st_teff        294
pl_rade         50
pl_bmasse       31
sy_dist         27
st_mass          9
disc_year        1
dtype: int64

Rows with ZERO missing values: 4110 (65.5%)
Rows with ≤2 missing values: 5691 (90.6%)


In [6]:
df_selected = df[columns_to_keep].copy()
 
print("\n" + "="*80)
print("SELECTED FEATURES (17 total)")
print("="*80)
print(f"New shape: {df_selected.shape}")
print(f"\nSelected columns:")
for i, col in enumerate(columns_to_keep, 1):
    print(f"  {i:2d}. {col}")


SELECTED FEATURES (17 total)
New shape: (6278, 15)

Selected columns:
   1. pl_name
   2. hostname
   3. pl_orbper
   4. pl_orbsmax
   5. pl_rade
   6. pl_bmasse
   7. pl_insol
   8. pl_eqt
   9. st_teff
  10. st_rad
  11. st_mass
  12. discoverymethod
  13. disc_year
  14. pl_controv_flag
  15. sy_dist


In [10]:
# Calculate completeness for each row
rows_missing_count = df_selected.isnull().sum(axis=1)
 
scenarios = [0, 1, 2, 3, 5]
 
print("\nHow many rows survive different missing-value thresholds?\n")
print(f"Total rows in dataset: {len(df_selected)}\n")
 
for threshold in scenarios:
    rows_passing = (rows_missing_count <= threshold).sum()
    pct_retained = (rows_passing / len(df_selected)) * 100
    print(f"  Rows with ≤{threshold} missing values: {rows_passing:,} ({pct_retained:5.1f}%)")


How many rows survive different missing-value thresholds?

Total rows in dataset: 6278

  Rows with ≤0 missing values: 4,110 ( 65.5%)
  Rows with ≤1 missing values: 4,656 ( 74.2%)
  Rows with ≤2 missing values: 5,691 ( 90.6%)
  Rows with ≤3 missing values: 5,969 ( 95.1%)
  Rows with ≤5 missing values: 6,265 ( 99.8%)


In [11]:
print("\n" + "="*80)
print("RECOMMENDATION FOR MISSING DATA HANDLING")
print("="*80)
 
threshold_recommended = 2
rows_recommended = (rows_missing_count <= threshold_recommended).sum()
pct_recommended = (rows_recommended / len(df_selected)) * 100
 
print(f"\nProposed strategy: Keep rows with ≤{threshold_recommended} missing values")
print(f"  → Retains {rows_recommended:,} rows ({pct_recommended:.1f}% of dataset)")
print(f"  → Loses {len(df_selected) - rows_recommended:,} rows ({100-pct_recommended:.1f}%)")
print(f"\nRationale:")
print(f"  • Preserves >90% of data while maintaining quality")
print(f"  • Allows flexibility for imputation strategies")
print(f"  • Balances sample size with feature richness")
 


RECOMMENDATION FOR MISSING DATA HANDLING

Proposed strategy: Keep rows with ≤2 missing values
  → Retains 5,691 rows (90.6% of dataset)
  → Loses 587 rows (9.4%)

Rationale:
  • Preserves >90% of data while maintaining quality
  • Allows flexibility for imputation strategies
  • Balances sample size with feature richness


In [13]:
# Filter to rows with ≤2 missing values
df_clean = df_selected[rows_missing_count <= threshold_recommended].copy()
 
# Save to processed data folder
output_path = '../data/clean/exoplanets_selected_features.csv'
df_clean.to_csv(output_path, index=False)
 
print(f"\nSaved to: {output_path}")
print(f"Final dataset shape: {df_clean.shape}")
print(f"\nThis dataset is ready for Phase 2 (EDA) and Phase 3 (Modeling)")


Saved to: ../data/clean/exoplanets_selected_features.csv
Final dataset shape: (5691, 15)

This dataset is ready for Phase 2 (EDA) and Phase 3 (Modeling)


In [ ]:
 
print("\n" + "="*80)
print("FEATURE STATISTICS (Preview)")
print("="*80)
 
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
 
print("\nNumeric features summary:")
print(df_clean[numeric_cols].describe().round(3))


FEATURE STATISTICS (Preview)

Numeric features summary:
          pl_orbper  pl_orbsmax   pl_rade  pl_bmasse   pl_insol    pl_eqt  \
count  5.657000e+03    5488.000  5680.000   5667.000   4400.000  4660.000   
mean   7.530098e+04       6.580     5.603    346.462    419.720   913.459   
std    5.347268e+06     161.419     5.191   1031.961   1287.673   458.186   
min    1.770000e-01       0.005     0.310      0.037      0.000    55.900   
25%    4.309000e+00       0.051     1.800      4.109     24.110   571.750   
50%    1.074000e+01       0.094     2.770      8.750     99.986   823.000   
75%    3.799300e+01       0.231    11.145    147.155    376.230  1162.310   
max    4.020000e+08    7506.000    33.600   9534.852  44900.000  4050.000   

         st_teff    st_rad   st_mass  disc_year  pl_controv_flag   sy_dist  
count   5690.000  5689.000  5690.000   5690.000         5691.000  5674.000  
mean    5355.539     1.509     0.961   2016.916            0.007   489.201  
std     1217.396  

In [15]:

print("\n" + "="*80)
print("CALCULATING MISSING VALUE IMPACT")
print("="*80)

# Analize the missing values in pl_insol and pl_eqt to understand the extent of the issue and its potential impact on our analysis.
missing_critical = df_selected[['pl_name', 'hostname', 'pl_insol', 'pl_eqt', 'pl_orbsmax', 'st_teff', 'st_rad']].copy()
missing_critical['insol_missing'] = missing_critical['pl_insol'].isnull()
missing_critical['eqt_missing'] = missing_critical['pl_eqt'].isnull()

print("Rows missing BOTH insol and eqt:")
both_missing = missing_critical[missing_critical['insol_missing'] & missing_critical['eqt_missing']]
print(f"Count: {len(both_missing)} ({100*len(both_missing)/len(df_selected):.1f}%)")

print("\nRows missing ONLY insol (have eqt):")
only_insol = missing_critical[missing_critical['insol_missing'] & ~missing_critical['eqt_missing']]
print(f"Count: {len(only_insol)} ({100*len(only_insol)/len(df_selected):.1f}%)")

print("\nRows missing ONLY eqt (have insol):")
only_eqt = missing_critical[missing_critical['eqt_missing'] & ~missing_critical['insol_missing']]
print(f"Count: {len(only_eqt)} ({100*len(only_eqt)/len(df_selected):.1f}%)")

print("\nRows with NEITHER missing (both values present):")
neither_missing = missing_critical[~missing_critical['insol_missing'] & ~missing_critical['eqt_missing']]
print(f"Count: {len(neither_missing)} ({100*len(neither_missing)/len(df_selected):.1f}%)")

# Example: show a few rows with missing values
print("\n\nSample of rows with missing values:")
print(missing_critical[missing_critical['insol_missing'] | missing_critical['eqt_missing']].head(10))


CALCULATING MISSING VALUE IMPACT
Rows missing BOTH insol and eqt:
Count: 1515 (24.1%)

Rows missing ONLY insol (have eqt):
Count: 363 (5.8%)

Rows missing ONLY eqt (have insol):
Count: 86 (1.4%)

Rows with NEITHER missing (both values present):
Count: 4314 (68.7%)


Sample of rows with missing values:
                   pl_name               hostname  pl_insol  pl_eqt  \
0                 11 Com b                 11 Com       NaN     NaN   
1                 11 UMi b                 11 UMi       NaN     NaN   
2                 14 And b                 14 And       NaN     NaN   
3                 14 Her b                 14 Her       NaN     NaN   
4               16 Cyg B b               16 Cyg B       NaN     NaN   
5                 17 Sco b                 17 Sco       NaN     NaN   
6                 18 Del b                 18 Del       NaN     NaN   
7  1RXS J160929.1-210524 b  1RXS J160929.1-210524       NaN  1700.0   
8                 24 Boo b                 24 Boo       N